# P1-29 · Experiment E3, external reference: CSAW-S test set with expert anatomy outlines (CPU, ~20–40 min)
CSAW-S (Matsoukas et al., ICML 2020; Zenodo 10.5281/zenodo.4030660; CC BY-NC-ND 4.0): screening mammograms from Stockholm. In the
test set the mammary gland and the pectoral muscle were outlined by expert annotators. Breast-plus-pectoral foreground = mammary gland
∪ pectoral muscle (as in Larroza et al., Diagnostics 2024); breast-only = mammary gland. Writes only `MyDrive/P1_v2/e3_csaws/`.

In [ ]:
# 1 · Drive, paths, download (Zenodo API)
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, glob, json, re, zipfile, tarfile, requests
import numpy as np, pandas as pd, cv2
ROOT = '/content/drive/MyDrive/QSEG_FINAL'; OUT = '/content/drive/MyDrive/P1_v2'; CS = f'{OUT}/e3_csaws'
for d in (f'{CS}/images', f'{CS}/sheets'): os.makedirs(d, exist_ok=True)
LOCAL = '/content/csaws'; os.makedirs(LOCAL, exist_ok=True)
sys.path.insert(0, ROOT + '/code')
from qseg.data_prep import resize_pair, chest_wall_side
rec = requests.get('https://zenodo.org/api/records/4030660', timeout=60).json()
print([(f['key'], round(f['size'] / 2 ** 20, 1)) for f in rec['files']])
for f in rec['files']:
    dst = f"{LOCAL}/{f['key']}"
    if not os.path.exists(dst):
        with requests.get(f['links'].get('self') or f['links'].get('download'), stream=True, timeout=1800) as r:
            r.raise_for_status()
            with open(dst, 'wb') as o:
                for ch in r.iter_content(1 << 20): o.write(ch)
    if dst.endswith('.zip') and not os.path.exists(dst + '.done'): zipfile.ZipFile(dst).extractall(LOCAL); open(dst + '.done', 'w').close()
    if dst.endswith(('.tar.gz', '.tgz', '.tar')) and not os.path.exists(dst + '.done'): tarfile.open(dst).extractall(LOCAL); open(dst + '.done', 'w').close()
TEST = [d for d in glob.glob(f'{LOCAL}/**/test_data', recursive=True)][0]
print('test_data:', TEST, '| subfolders:', sorted(os.listdir(TEST)))
for a in sorted(os.listdir(TEST)):
    p = os.path.join(TEST, a)
    if os.path.isdir(p):
        fs = glob.glob(f'{p}/**/*.png', recursive=True); print(a, len(fs), 'png, e.g.', [os.path.relpath(x, p) for x in fs[:4]])

In [ ]:
# 2 · Collect images and the expert outlines of the mammary gland and the pectoral muscle
CLS = ('mammary_gland', 'pectoral_muscle')
pat = re.compile(r'^(?P<sid>.+?)_(?P<cls>' + '|'.join(['cancer', 'calcifications', 'axillary_lymph_nodes', 'thick_vessels', 'foreign_object', 'skin',
                                                         'nipple', 'text', 'non-mammary_tissue', 'pectoral_muscle', 'mammary_gland']) + r')\.png$')
imgs = {}
for p in glob.glob(f'{TEST}/anonymized_dataset/**/*.png', recursive=True):
    b = os.path.basename(p)
    if pat.match(b): continue
    sid = b[:-4].replace('_8bit', '')
    if b.endswith('_8bit.png') or sid not in imgs: imgs[sid] = p
ann = {}
for a in sorted(d for d in os.listdir(TEST) if d.startswith('annotator')):
    for p in glob.glob(f'{TEST}/{a}/**/*.png', recursive=True):
        m = pat.match(os.path.basename(p))
        if m and m['cls'] in CLS: ann.setdefault((a, m['sid']), {})[m['cls']] = p
print('test images:', len(imgs), '| annotator x image with gland/pectoral outlines:',
      pd.Series([a for a, _ in ann]).value_counts().to_dict())

In [ ]:
# 3 · Masks at 256 x 256 (Paper 1 preprocessing) and agreement between the expert annotators
def read_mask(p, shape):
    m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if m is None: return np.zeros(shape, bool)
    if m.shape != shape: m = cv2.resize(m, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return m > 0
def dice(a, b):
    s = a.sum() + b.sum(); return 1.0 if s == 0 else 2 * (a & b).sum() / s
rows, M = [], {}
for sid, ip in sorted(imgs.items()):
    g = cv2.imread(ip, cv2.IMREAD_GRAYSCALE)
    if g is None: continue
    if g.dtype != np.uint8: g = cv2.normalize(g, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    have = [a for (a, s) in ann if s == sid and all(c in ann[(a, s)] for c in CLS)]
    if not have: continue
    for a in have:
        gl = read_mask(ann[(a, sid)]['mammary_gland'], g.shape); pe = read_mask(ann[(a, sid)]['pectoral_muscle'], g.shape)
        full = gl | pe
        img_s, full_s = resize_pair(g, (full * 255).astype(np.uint8), 256, True)
        side = 'left' if chest_wall_side((full * 255).astype(np.uint8)) == 'right' else 'right'
        from qseg.data_prep import pad_to_square
        br_s = cv2.resize(pad_to_square((gl & ~pe).astype(np.uint8) * 255, side), (256, 256), interpolation=cv2.INTER_NEAREST) > 127
        fid = 'c_' + re.sub(r'[^A-Za-z0-9]', '', sid)
        os.makedirs(f'{CS}/{a}/full', exist_ok=True); os.makedirs(f'{CS}/{a}/breast', exist_ok=True)
        cv2.imwrite(f'{CS}/images/{fid}.png', img_s); cv2.imwrite(f'{CS}/{a}/full/{fid}.png', full_s); cv2.imwrite(f'{CS}/{a}/breast/{fid}.png', br_s.astype(np.uint8) * 255)
        M[(a, fid)] = (full_s > 127, br_s)
        rows.append(dict(file_id=fid, source=sid, patient_id=sid.split('_')[0], annotator=a, has_pectoral=bool(pe.any()), frac_full=float((full_s > 127).mean())))
man = pd.DataFrame(rows); man.to_csv(f'{CS}/manifest.csv', index=False)
print('images with outlines:', man.file_id.nunique(), '| patients:', man.patient_id.nunique(), '| per annotator:', man.annotator.value_counts().to_dict())
anns = sorted(man.annotator.unique()); agr = []
for i in range(len(anns)):
    for j in range(i + 1, len(anns)):
        for f in man.file_id.unique():
            if (anns[i], f) in M and (anns[j], f) in M:
                agr.append(dict(pair=f'{anns[i]} vs {anns[j]}', file_id=f, dice_full=dice(M[(anns[i], f)][0], M[(anns[j], f)][0]), dice_breast=dice(M[(anns[i], f)][1], M[(anns[j], f)][1])))
agr = pd.DataFrame(agr); agr.to_csv(f'{CS}/annotator_agreement.csv', index=False)
print(agr.groupby('pair')[['dice_full', 'dice_breast']].describe().round(3).to_string() if len(agr) else 'no pairs')

In [ ]:
# 4 · Overlay sheet (expert outlines on the 256 x 256 inputs; green = breast + pectoral, orange = breast only)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
a0 = man.annotator.value_counts().index[0]; ids = man[man.annotator == a0].file_id.tolist()
n = len(ids); cols = 8; rws = int(np.ceil(n / cols))
fig, ax = plt.subplots(rws, cols, figsize=(16, 2.2 * rws), squeeze=False)
for a, f in zip(ax.ravel(), ids):
    im = cv2.cvtColor(cv2.imread(f'{CS}/images/{f}.png', 0), cv2.COLOR_GRAY2RGB)
    for k, col in ((0, (60, 220, 60)), (1, (255, 140, 0))):
        cs, _ = cv2.findContours(M[(a0, f)][k].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(im, cs, -1, col, 1)
    a.imshow(im); a.set_title(f, fontsize=6)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.savefig(f'{CS}/sheets/csaws_test_{a0}.png', dpi=90); plt.close()
print('DONE')